# 16 — Demand Sensing
## SunnyBest Retail Forecasting System

> **What is demand sensing?**  
> Weekly forecasts tell you what to expect over 7 days. Demand sensing tells you **right now** — within the current week — whether actual demand is running hotter or colder than expected, so you can act before a stockout happens or before dead stock piles up.

### The problem it solves
Your weekly model forecasts Monday → Sunday in advance. But by Wednesday, you have 3 days of real sales. If those 3 days already show 80% of the expected weekly total, you have a demand spike — and you need to restock **now**, not next Saturday.

### Techniques used
| Technique | What it detects |
|-----------|----------------|
| **Rolling z-score** | How many standard deviations is today's sales from the historical average for this day? |
| **Week-to-date vs forecast** | What % of the weekly forecast has already been consumed by mid-week? |
| **CUSUM** | Cumulative sum of deviations — detects sustained shifts, not just one-off spikes |
| **Day-of-week baseline** | Compares Monday to historical Mondays, not to the overall daily average |

---
## 0. Setup

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
from sqlalchemy import create_engine
from urllib.parse import quote_plus

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None

plt.rcParams["figure.figsize"]    = (14, 5)
plt.rcParams["axes.spines.top"]   = False
plt.rcParams["axes.spines.right"] = False
PALETTE = ["#4C72B0","#C44E52","#55A868","#DD8452","#8172B2","#64B5CD","#CCB974"]

# ── Tunable parameters ────────────────────────────────────
ZSCORE_ALERT    = 2.0    # flag if z-score > this (2 = unusual, 3 = very unusual)
WTD_ALERT_PCT   = 0.70   # flag if week-to-date consumption > 70% of weekly forecast by mid-week
HISTORY_WEEKS   = 12     # weeks of history to build the baseline
CUSUM_THRESHOLD = 5.0    # CUSUM alert threshold (in units)

# ── DB connection ─────────────────────────────────────────
host     = "aws-1-eu-central-1.pooler.supabase.com"
port     = 5432
database = "postgres"
user     = "postgres.ogkdfmkybqtrsglcizzt"
password = quote_plus(os.getenv("SUPABASE_DB_PASSWORD", "Bonabosssfs01"))

engine = create_engine(
    f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}",
    pool_pre_ping=True, pool_recycle=300
)

print("Setup complete")

---
## 1. Load Daily Sales & Enrich

> We need daily granularity — not weekly — to sense what's happening mid-week.

In [ ]:
df = pd.read_sql("""
    SELECT
        s.date,
        s.store_id,
        st.store_name,
        st.store_size,
        s.product_id,
        p.product_name,
        p.category,
        s.units_sold,
        s.stockout_occurred,
        c.day_of_week,
        c.day_of_week_num,
        c.is_weekend,
        c.is_holiday,
        c.is_payday,
        c.season
    FROM core.fact_sales s
    LEFT JOIN core.dim_stores   st ON s.store_id   = st.store_id
    LEFT JOIN core.dim_products  p ON s.product_id  = p.product_id
    LEFT JOIN core.dim_calendar  c ON s.date        = c.date
    ORDER BY s.date ASC
""", engine)

df["date"] = pd.to_datetime(df["date"])
df["week_start"] = df["date"] - pd.to_timedelta(df["day_of_week_num"], unit="D")

print(f"Rows      : {len(df):,}")
print(f"Date range: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Stores    : {df['store_id'].nunique()}")
print(f"Products  : {df['product_id'].nunique()}")

---
## 2. Build Day-of-Week Baseline

> Monday is always different from Friday. We compare each day to its own historical average.  
> Using the last N weeks of history to keep the baseline current.

In [ ]:
latest_date   = df["date"].max()
history_start = latest_date - pd.Timedelta(weeks=HISTORY_WEEKS)

history = df[(df["date"] >= history_start) & (df["date"] < latest_date - pd.Timedelta(days=6))].copy()

# Baseline: mean and std of units_sold per store × product × day_of_week
baseline = (history.groupby(["store_id", "product_id", "day_of_week_num"])
              .agg(
                  baseline_mean =("units_sold", "mean"),
                  baseline_std  =("units_sold", "std"),
                  baseline_obs  =("units_sold", "count"),
              )
              .reset_index())

# Avoid division by zero — use small std floor
baseline["baseline_std"] = baseline["baseline_std"].fillna(0).clip(lower=0.1)

print(f"Baseline built from {history_start.date()} → {history['date'].max().date()}")
print(f"Baseline rows: {len(baseline):,}")
display(baseline.head(10))

---
## 3. Rolling Z-Score — Is Today Unusual?

> **Z-score** = (actual - historical mean for this day) / historical std  
> Z > 2 → unusually high demand (potential stockout risk)  
> Z < -2 → unusually low demand (potential dead stock risk)

In [ ]:
# Last 14 days of data
recent = df[df["date"] > latest_date - pd.Timedelta(days=14)].copy()

recent = recent.merge(baseline, on=["store_id", "product_id", "day_of_week_num"], how="left")
recent["z_score"] = ((recent["units_sold"] - recent["baseline_mean"]) / recent["baseline_std"]).round(3)

recent["signal"] = "Normal"
recent.loc[recent["z_score"] >  ZSCORE_ALERT, "signal"] = "DEMAND SPIKE"
recent.loc[recent["z_score"] < -ZSCORE_ALERT, "signal"] = "DEMAND DROP"

spikes = recent[recent["signal"] == "DEMAND SPIKE"].sort_values("z_score", ascending=False)
drops  = recent[recent["signal"] == "DEMAND DROP"].sort_values("z_score")

print(f"Total observations (last 14 days): {len(recent):,}")
print(f"Demand SPIKES (z > {ZSCORE_ALERT}): {len(spikes):,}")
print(f"Demand DROPS  (z < -{ZSCORE_ALERT}): {len(drops):,}")

print("\nTop 15 demand spikes:")
display(spikes[["date","store_name","product_name","category","units_sold","baseline_mean","z_score"]].head(15).reset_index(drop=True))

print("\nTop 15 demand drops:")
display(drops[["date","store_name","product_name","category","units_sold","baseline_mean","z_score"]].head(15).reset_index(drop=True))

---
## 4. Z-Score Distribution — How Unusual Is Recent Demand?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Z-score histogram
axes[0].hist(recent["z_score"].dropna(), bins=60, color=PALETTE[0], edgecolor="white")
axes[0].axvline( ZSCORE_ALERT, color=PALETTE[1], ls="--", lw=1.5, label=f"+{ZSCORE_ALERT} spike threshold")
axes[0].axvline(-ZSCORE_ALERT, color=PALETTE[2], ls="--", lw=1.5, label=f"-{ZSCORE_ALERT} drop threshold")
axes[0].axvline(0, color="grey", ls="-", lw=0.8)
axes[0].set_title("Z-Score Distribution (last 14 days)")
axes[0].set_xlabel("Z-Score")
axes[0].set_ylabel("Count")
axes[0].legend()

# Spikes + drops by store
alert_by_store = (recent[recent["signal"] != "Normal"]
                  .groupby(["store_name","signal"])
                  .size().reset_index(name="count"))
if not alert_by_store.empty:
    pivot_alerts = alert_by_store.pivot(index="store_name", columns="signal", values="count").fillna(0)
    pivot_alerts.plot(kind="bar", ax=axes[1], color=[PALETTE[1], PALETTE[2]])
    axes[1].set_title("Demand Signals by Store (last 14 days)")
    axes[1].set_ylabel("Alert count")
    axes[1].tick_params(axis="x", rotation=30)
    axes[1].legend(title="Signal")
else:
    axes[1].text(0.5, 0.5, "No alerts in last 14 days", ha="center", va="center", transform=axes[1].transAxes)

plt.tight_layout()
plt.show()

---
## 5. Week-to-Date vs Weekly Forecast

> How much of the weekly forecast has already been consumed?  
> If it's Wednesday and you've used 70%+ of the forecast — restock now, not Saturday.

In [ ]:
FORECAST_PATH = "../data/outputs/weekly_forecasts.csv"

if os.path.exists(FORECAST_PATH):
    forecasts = pd.read_csv(FORECAST_PATH)
    forecasts["week_start"]       = pd.to_datetime(forecasts["week_start"])
    forecasts["store_id"]         = forecasts["store_id"].astype(str)
    forecasts["product_id"]       = pd.to_numeric(forecasts["product_id"], errors="coerce").astype(int)
    forecasts["predicted_units"]  = pd.to_numeric(forecasts["predicted_units"], errors="coerce")

    # Current week start (Monday)
    today      = df["date"].max()
    week_start = today - pd.Timedelta(days=today.weekday())

    # WTD actuals
    wtd = (df[df["date"] >= week_start]
             .groupby(["store_id","product_id"])
             .agg(wtd_units=("units_sold","sum"))
             .reset_index())

    # Latest forecast for this week
    fc_week = forecasts[forecasts["week_start"] <= week_start].sort_values("week_start").groupby(["store_id","product_id"]).last().reset_index()

    wtd_vs_fc = wtd.merge(fc_week[["store_id","product_id","predicted_units"]], on=["store_id","product_id"], how="left")
    wtd_vs_fc["pct_consumed"] = (wtd_vs_fc["wtd_units"] / wtd_vs_fc["predicted_units"].replace(0, np.nan) * 100).round(1)
    wtd_vs_fc["days_elapsed"] = (today - week_start).days + 1
    wtd_vs_fc["alert"]        = wtd_vs_fc["pct_consumed"] > WTD_ALERT_PCT * 100

    alerts = wtd_vs_fc[wtd_vs_fc["alert"]].sort_values("pct_consumed", ascending=False)

    print(f"Current week start : {week_start.date()}")
    print(f"Days elapsed       : {wtd_vs_fc['days_elapsed'].iloc[0]} of 7")
    print(f"WTD alert threshold: >{WTD_ALERT_PCT:.0%} of weekly forecast consumed")
    print(f"Products on alert  : {len(alerts):,}")

    if not alerts.empty:
        alerts = alerts.merge(
            df[["store_id","product_id","store_name","product_name","category"]].drop_duplicates(),
            on=["store_id","product_id"], how="left"
        )
        print("\nProducts that have consumed >70% of weekly forecast already:")
        display(alerts[["store_name","product_name","category","wtd_units","predicted_units","pct_consumed"]].head(20).reset_index(drop=True))
else:
    print("No forecast file found. Run generate_weekly_forecast.py first.")

---
## 6. CUSUM — Detecting Sustained Demand Shifts

> A single spike could be noise. CUSUM catches when demand has **consistently** been above or below baseline for several days in a row.  
> This is the difference between a one-off event and a genuine demand shift that requires a model retrain.

In [ ]:
# Aggregate to store × category level for cleaner CUSUM signal
daily_cat = (df.groupby(["date","store_name","category"])
               .agg(units_sold=("units_sold","sum"))
               .reset_index())

# Build baseline per store × category × day_of_week
daily_cat["day_of_week_num"] = daily_cat["date"].dt.weekday
hist_cat = daily_cat[daily_cat["date"] < latest_date - pd.Timedelta(days=14)]
base_cat = (hist_cat.groupby(["store_name","category","day_of_week_num"])
              .agg(base_mean=("units_sold","mean"), base_std=("units_sold","std"))
              .reset_index())
base_cat["base_std"] = base_cat["base_std"].fillna(1.0).clip(lower=0.5)

recent_cat = daily_cat[daily_cat["date"] > latest_date - pd.Timedelta(days=21)].copy()
recent_cat = recent_cat.merge(base_cat, on=["store_name","category","day_of_week_num"], how="left")
recent_cat["deviation"] = recent_cat["units_sold"] - recent_cat["base_mean"].fillna(0)

# Compute CUSUM per store × category
cusum_rows = []
for (store, cat), grp in recent_cat.groupby(["store_name","category"]):
    grp = grp.sort_values("date")
    cusum = grp["deviation"].cumsum().values
    for i, row in enumerate(grp.itertuples()):
        cusum_rows.append({
            "date":       row.date,
            "store_name": store,
            "category":   cat,
            "units_sold": row.units_sold,
            "base_mean":  row.base_mean,
            "deviation":  row.deviation,
            "cusum":      cusum[i],
        })

cusum_df = pd.DataFrame(cusum_rows)

# Latest CUSUM per store × category
latest_cusum = cusum_df.groupby(["store_name","category"]).last().reset_index()
latest_cusum["cusum_alert"] = latest_cusum["cusum"].abs() > CUSUM_THRESHOLD
latest_cusum = latest_cusum.sort_values("cusum", ascending=False)

print(f"CUSUM alert threshold: ±{CUSUM_THRESHOLD} units")
print(f"Store-category combinations on CUSUM alert: {latest_cusum['cusum_alert'].sum()}")
print()
display(latest_cusum[["store_name","category","cusum","cusum_alert"]].reset_index(drop=True))

---
## 7. CUSUM Trend Charts — Top Alerted Categories

In [ ]:
alerted = latest_cusum[latest_cusum["cusum_alert"]].head(6)

if len(alerted) == 0:
    print("No CUSUM alerts — demand is tracking close to baseline. Model is healthy.")
else:
    n = len(alerted)
    fig, axes = plt.subplots(1, min(n, 3), figsize=(16, 5))
    if n == 1:
        axes = [axes]

    for i, (_, row) in enumerate(alerted.head(3).iterrows()):
        sub = cusum_df[
            (cusum_df["store_name"] == row["store_name"]) &
            (cusum_df["category"]   == row["category"])
        ]
        ax = axes[i]
        ax.plot(sub["date"], sub["cusum"], color=PALETTE[1] if row["cusum"] > 0 else PALETTE[0], lw=2)
        ax.axhline(0,  color="grey", lw=0.8)
        ax.axhline( CUSUM_THRESHOLD, color=PALETTE[1], ls="--", lw=1, label=f"+{CUSUM_THRESHOLD}")
        ax.axhline(-CUSUM_THRESHOLD, color=PALETTE[0], ls="--", lw=1, label=f"-{CUSUM_THRESHOLD}")
        ax.set_title(f"{row['store_name']}\n{row['category']}")
        ax.set_ylabel("CUSUM")
        ax.tick_params(axis="x", rotation=30)
        ax.legend(fontsize=7)

    fig.suptitle("CUSUM — Sustained Demand Shifts (above zero = consistently above baseline)", fontsize=11)
    plt.tight_layout()
    plt.show()

---
## 8. Demand Sensing Dashboard — Single Summary View

> One table that combines all signals. This is what a store manager or supply chain team would see every morning.

In [ ]:
# Aggregate z-scores to store × category level for last 3 days
last3 = recent[recent["date"] >= latest_date - pd.Timedelta(days=3)]
dashboard = (last3.groupby(["store_name","category"])
               .agg(
                   avg_z_score   =("z_score","mean"),
                   max_z_score   =("z_score","max"),
                   spike_count   =("signal",lambda x: (x == "DEMAND SPIKE").sum()),
                   drop_count    =("signal",lambda x: (x == "DEMAND DROP").sum()),
                   actual_units  =("units_sold","sum"),
                   stockouts     =("stockout_occurred","sum"),
               )
               .reset_index())

dashboard = dashboard.merge(
    latest_cusum[["store_name","category","cusum","cusum_alert"]],
    on=["store_name","category"], how="left"
)

dashboard["avg_z_score"] = dashboard["avg_z_score"].round(2)
dashboard["cusum"]       = dashboard["cusum"].round(2)

def overall_signal(row):
    if row["spike_count"] > 0 or row["cusum_alert"]:
        return "⚠️ WATCH"
    if row["drop_count"] > 0:
        return "📉 LOW"
    return "✅ Normal"

dashboard["overall"] = dashboard.apply(overall_signal, axis=1)
dashboard = dashboard.sort_values(["overall","max_z_score"], ascending=[True, False])

print(f"Demand Sensing Dashboard — based on last 3 days of data (up to {latest_date.date()})")
print()
display(dashboard[["store_name","category","avg_z_score","max_z_score",
                    "spike_count","drop_count","stockouts","cusum","overall"]].reset_index(drop=True))

---
## Insights

**What demand sensing tells you that weekly forecasting cannot:**
- A weekly forecast says "expect 100 units this week" — demand sensing says "you've already sold 80 by Wednesday"
- A z-score spike on Tuesday morning gives you 4 days to restock before the weekend rush
- CUSUM sustained above baseline for 5+ days means this isn't noise — the demand pattern has genuinely shifted and the model may need retraining

**How to act on the signals:**

| Signal | What it means | Action |
|--------|--------------|--------|
| Z-score > 2 | Unusual spike today vs historical same-day | Check inventory — consider emergency restock |
| WTD > 70% by mid-week | Forecast will be exceeded | Bring forward next restock order |
| CUSUM persistently positive | Demand has structurally shifted upward | Retrain model with recent data |
| CUSUM persistently negative | Demand has structurally shifted downward | Reduce safety stock targets |
| Z-score < -2 | Unusual demand drop | Investigate — competitor promotion? Stockout cascade? |